# SST-2 Order-Sensitivity Experiment

This notebook runs the full proposal workflow:

- install dependencies
- verify GPU availability
- optionally log into Hugging Face
- fine-tune 10 DistilBERT runs with different data orders
- analyze disagreement, confidence variance, and stability scoring
- inspect the generated outputs


## Recommended Runtime

Use a **GPU** runtime. CPU will work, but 10 fine-tuning runs will be much slower. The SST-2 dataset and `distilbert-base-uncased` are both public, so Hugging Face login is optional unless you want authenticated access or higher rate limits.

In [ ]:
import os
import pathlib
import subprocess
import sys

CWD = pathlib.Path.cwd()
print('Working directory:', CWD)
print('Python:', sys.version)


In [ ]:
required_files = [
    'requirements.txt',
    'train_sst2.py',
    'run_experiment.py',
    'analyze_runs.py',
    'src/ur2phd/experiment.py',
]

def find_project_root(start_path: pathlib.Path) -> pathlib.Path | None:
    candidates = [start_path] + list(start_path.parents)
    for candidate in candidates:
        if all((candidate / rel_path).exists() for rel_path in required_files):
            return candidate
    return None

PROJECT_ROOT = find_project_root(CWD)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate the project root from the current Colab working directory. '
        'Open the notebook from the repo folder, or set PROJECT_ROOT manually to the folder '
        'containing requirements.txt and train_sst2.py.'
    )

print('Project root:', PROJECT_ROOT)
os.chdir(PROJECT_ROOT)


In [ ]:
subprocess.run([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-r',
    'requirements.txt',
], check=True)


In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. You can still run this on CPU, but it will be slower.')


## Optional: Hugging Face Login

You do **not** need a Hugging Face account to download `distilbert-base-uncased` or GLUE SST-2 because both are public.

If you want to log in anyway, create a token at https://huggingface.co/settings/tokens and run the next cell.

In [ ]:
# Uncomment if you want to log in.
# from huggingface_hub import notebook_login
# notebook_login()


## Quick Check: Load SST-2

This confirms the Hugging Face Datasets loader can download the GLUE SST-2 split.

In [ ]:
from datasets import load_dataset

sst2 = load_dataset('glue', 'sst2')
sst2


## Experiment Settings

If you are just testing the pipeline, set `RUNS = 2` first. For the full project result, use `RUNS = 10`.

In [ ]:
RUNS = 10
BASE_PERMUTATION_SEED = 100
MODEL_SEED = 13
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 2e-5
MAX_LENGTH = 128
USE_FP16 = torch.cuda.is_available()

print({
    'runs': RUNS,
    'base_permutation_seed': BASE_PERMUTATION_SEED,
    'model_seed': MODEL_SEED,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'max_length': MAX_LENGTH,
    'fp16': USE_FP16,
})


## Run Training Sweep

In [ ]:
command = [
    sys.executable,
    'run_experiment.py',
    '--runs', str(RUNS),
    '--base-permutation-seed', str(BASE_PERMUTATION_SEED),
    '--model-seed', str(MODEL_SEED),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--learning-rate', str(LEARNING_RATE),
    '--max-length', str(MAX_LENGTH),
]

if USE_FP16:
    command.append('--fp16')

print('Running:', ' '.join(command))
subprocess.run(command, check=True)


## Analyze Runs

In [ ]:
subprocess.run([
    sys.executable,
    'analyze_runs.py',
    '--artifacts-root', 'artifacts',
    '--output-dir', 'analysis',
], check=True)


## Inspect Summary Outputs

In [ ]:
import json
import pandas as pd

summary = json.loads((PROJECT_ROOT / 'analysis' / 'summary.json').read_text(encoding='utf-8'))
summary


In [ ]:
model_scores = pd.read_csv(PROJECT_ROOT / 'analysis' / 'model_scores.csv')
model_scores


In [ ]:
top_unstable = pd.read_csv(PROJECT_ROOT / 'analysis' / 'top_unstable_sentences.csv')
top_unstable.head(20)


In [ ]:
from IPython.display import Image, display

for plot_name in [
    'disagreement_histogram.png',
    'pairwise_agreement_heatmap.png',
    'confidence_variance_histogram.png',
    'agreement_vs_accuracy.png',
]:
    print(plot_name)
    display(Image(filename=str(PROJECT_ROOT / 'analysis' / plot_name)))


## Files You’ll Use In The Writeup

- `analysis/summary.json`
- `analysis/model_scores.csv`
- `analysis/sentence_summary.csv`
- `analysis/top_unstable_sentences.csv`
- plots under `analysis/`
